# DermaCare AI – train the skin lesion classifier (Google Colab)

This notebook trains the 7-class skin lesion model used by the DermaCare AI Engine on the **HAM10000** dataset
(Tschandl et al., 2018) with transfer learning, then evaluates it and gives you the `.pt` file to upload in
**Admin → AI model versions**.

**Before you start:** `Runtime → Change runtime type → GPU (T4)`. Training EfficientNet-B0 for 15 epochs takes roughly 25–40 minutes.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch; print('CUDA available:', torch.cuda.is_available())

## 2. Get the project code
Option A: zip the `ai-service` folder on your computer and upload it here.  
Option B: if your project is on GitHub, clone it instead (edit the URL).

In [ ]:
# Option A: upload ai-service.zip
from google.colab import files
uploaded = files.upload()          # choose ai-service.zip
!unzip -q -o ai-service.zip -d /content/
%cd /content/ai-service

# Option B (instead of A):
# !git clone https://github.com/YOUR-USER/YOUR-REPO.git /content/repo
# %cd /content/repo/ai-service

In [ ]:
!pip install -q timm>=0.9 scikit-learn opencv-python-headless

## 3. Download HAM10000
The easiest way is the Kaggle copy (*Skin Cancer MNIST: HAM10000*, `kmader/skin-cancer-mnist-ham10000`):
1. On kaggle.com go to **Settings → API → Create New Token**; this downloads `kaggle.json`.
2. Run the next cell and upload `kaggle.json`.

Alternative: download the original files from the Harvard Dataverse (doi:10.7910/DVN/DBW86T), upload them to
Google Drive and unzip them into `/content/ham10000`.

In [ ]:
from google.colab import files
files.upload()                     # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content
!unzip -q -o /content/skin-cancer-mnist-ham10000.zip -d /content/ham10000
!ls /content/ham10000 | head

## 4. Look at the data (class imbalance)

In [ ]:
import pandas as pd
meta = pd.read_csv('/content/ham10000/HAM10000_metadata.csv')
print(len(meta), 'images of', meta.lesion_id.nunique(), 'lesions')
print(meta.dx.value_counts())
meta.dx.value_counts().plot.bar(title='HAM10000 class distribution');

## 5. (Optional) Calibrate the image quality check
Shows blur / brightness / contrast on real dermoscopic images, so the quality check does not reject good photos.

In [ ]:
!python training/calibrate_quality.py --images /content/ham10000 --sample 400

## 6. Train
* split by lesion (70 / 15 / 15), class-weighted loss, augmentation, best epoch chosen by validation balanced accuracy
* change `--arch` to compare models, e.g. `resnet50` (baseline) or `efficientnet_b3` (bigger, use `--img-size 300`)

In [ ]:
!python training/train.py --data-dir /content/ham10000 --arch efficientnet_b0 --epochs 15 --batch-size 32 --out /content/models --name dermacare_effnetb0_v1

## 7. Results

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open('/content/models/dermacare_effnetb0_v1_metrics.json'))
for k in ['accuracy', 'balanced_accuracy', 'macro_f1', 'roc_auc_macro', 'melanoma_sensitivity', 'melanoma_specificity', 'melanoma_threshold']:
    print(f'{k:24s} {m[k]}')
print()
print(pd.DataFrame(m['per_class']).T)
display(Image('/content/models/dermacare_effnetb0_v1_training_curves.png'))
display(Image('/content/models/dermacare_effnetb0_v1_confusion_matrix.png'))

## 8. (Optional) Baseline for your report
Train a ResNet-50 with the same split and compare it with EfficientNet-B0 in a table.

In [ ]:
!python training/train.py --data-dir /content/ham10000 --arch resnet50 --epochs 15 --out /content/models --name dermacare_resnet50_v1

## 9. Download the model

In [ ]:
from google.colab import files
files.download('/content/models/dermacare_effnetb0_v1.pt')
files.download('/content/models/dermacare_effnetb0_v1_metrics.json')

Then, in DermaCare AI: **Admin → AI & chatbot → AI model versions** → upload the `.pt` and the metrics file → **Activate**.
(Or copy the `.pt` into `ai-service/models/` and restart the AI Engine.)

## 10. (Optional) Evaluate the FAQ chatbot

In [ ]:
# needs database/data/faqs.json from the Laravel project; upload it if it is not in the zip
!python training/chatbot_eval.py --faqs training/faqs.json --engine tfidf
!pip install -q sentence-transformers
!python training/chatbot_eval.py --faqs training/faqs.json --engine sbert